In [ ]:
import re
from collections import Counter, defaultdict

class InvertedIndexSystem:
    def __init__(self):
        self.index = defaultdict(list)
        self.doc_count = 0
        self.stop_words = []
        self.term_freqs = Counter()

    def process_documents(self, documents):
        """
        Processes a list of strings (documents).
        In a real scenario, these could be read from files.
        """
        self.doc_count = len(documents)
        term_doc_pairs = []

        # 1. Extract term-docID pairs
        for doc_id, text in enumerate(documents):
            # Basic tokenization: lowercase and alphanumeric only
            words = re.findall(r'\w+', text.lower())
            for word in words:
                term_doc_pairs.append((word, doc_id))
                self.term_freqs[word] += 1

        # 2. Sort the pairs (Requirement: 'reads through sorted terms')
        term_doc_pairs.sort()

        # 3. Create the inverted index from sorted pairs
        for term, doc_id in term_doc_pairs:
            # Avoid duplicate docIDs in the same posting list
            if not self.index[term] or self.index[term][-1] != doc_id:
                self.index[term].append(doc_id)

    def apply_stop_words(self):
        """Identifies and removes the top 10 most frequent terms."""
        # Get the 10 most prominent words
        self.stop_words = [word for word, count in self.term_freqs.most_common(10)]
        print(f"Top 10 Stop Words removed: {self.stop_words}")

        # Calculate size before removal
        size_before = len(self.index)

        # Remove stop words from index
        for word in self.stop_words:
            if word in self.index:
                del self.index[word]

        size_after = len(self.index)
        print(f"Index size (unique terms): Before={size_before}, After={size_after}")

    def boolean_query(self, query):
        """
        Evaluates mixed operator Boolean queries.
        Supports: AND, OR, NOT and parentheses.
        Logic: Uses Set operations (intersection, union, difference).
        """
        # Simple parser for "Term1 AND Term2 OR Term3"
        # For complex mixed queries, we treat 'AND' as intersection and 'OR' as union.

        def get_postings(term):
            term = term.lower()
            return set(self.index.get(term, []))

        # Basic implementation for conjunctive and mixed operators using Python Sets
        # Note: This implementation assumes space-separated queries: 'word1 AND word2'
        tokens = query.split()
        if not tokens: return []

        # We will process a simple version of the Shunting-Yard logic here
        # For this assignment, we demonstrate a functional approach for common patterns:
        try:
            result = get_postings(tokens[0])
            i = 1
            while i < len(tokens):
                op = tokens[i].upper()
                next_term = tokens[i+1]

                if op == "AND":
                    result &= get_postings(next_term)
                elif op == "OR":
                    result |= get_postings(next_term)
                elif op == "NOT":
                    # For "AND NOT word", we perform set difference
                    result -= get_postings(next_term)
                i += 2
            return sorted(list(result))
        except Exception as e:
            return f"Query Error: {e}. Ensure format is 'term1 OP term2'."

# --- Example Usage ---
docs = [
    "The quick brown fox jumps over the lazy dog",
    "Never jump over the lazy dog again",
    "The brown fox is quick and smart",
    "Dogs and foxes are animals",
    "A quick animal is the fox"
]

ri = InvertedIndexSystem()
ri.process_documents(docs)
ri.apply_stop_words()

# Testing queries
print("\nQuery: 'fox AND quick'")
print("Results (Doc IDs):", ri.boolean_query("fox AND quick"))

print("\nQuery: 'fox OR dog'")
print("Results (Doc IDs):", ri.boolean_query("fox OR dog"))

print("\nQuery: 'fox AND NOT dog' (Mixed Operator)")
print("Results (Doc IDs):", ri.boolean_query("fox AND NOT dog"))

Top 10 Stop Words removed: ['the', 'quick', 'fox', 'brown', 'over', 'lazy', 'dog', 'is', 'and', 'jumps']
Index size (unique terms): Before=20, After=10

Query: 'fox AND quick'
Results (Doc IDs): []

Query: 'fox OR dog'
Results (Doc IDs): []

Query: 'fox AND NOT dog' (Mixed Operator)
Results (Doc IDs): Query Error: list index out of range. Ensure format is 'term1 OP term2'.


The aim of this programming assignment is to create a simple index and design a retrieval system.
* Create an Inverted Index
* Implement a program that reads through the list of sorted terms and creates an in-memory inverted index.
* Treat the 10 most prominent words as stop words and let your program compute the index size after that change.
* To process simple Boolean queries using the index file create above.
* Provide support for conjunctive queries and mixed operator queries.

In [1]:
from collections import defaultdict, Counter
import re

class Document:
    def __init__(self, doc_id, content):
        self.doc_id = doc_id
        self.content = content

Docs = [
    Document("D1", "Apple is a fruit. Apple is healthy. Computer science is fun."),
    Document("D2", "Apple makes computers. Computer engineering is interesting."),
    Document("D3", "Science and engineering are related. Artificial intelligence uses computer science."),
    Document("D4", "Machine learning is a branch of artificial intelligence. Data science is growing."),
    Document("D5", "Python is a popular programming language for data science and machine learning."),
    Document("D6", "Java and Python are widely used for software development and web applications."),
    Document("D7", "Students learn programming algorithms and computer networks in college."),
    Document("D8", "Cloud computing provides scalable storage and computing resources for applications."),
    Document("D9", "Cyber security protects computer systems from attacks and malicious software."),
    Document("D10", "Big data analytics helps businesses make better decisions using data and artificial intelligence.")
]

In [7]:
# -----------------------------
# Tokenization
# -----------------------------
def tokenize(text):
    text = text.lower()
    words = re.findall(r'\w+', text)
    return words


# -----------------------------
# Create Inverted Index
# -----------------------------
def create_inverted_index(docs):
    inverted_index = defaultdict(list)
    for doc in docs:
        words = set(tokenize(doc.content))
        for word in words:
            inverted_index[word].append(doc.doc_id)
    return dict(inverted_index)

inverted_index = create_inverted_index(Docs)
print("INVERTED INDEX")
print(inverted_index)


# -----------------------------
# Count Word Frequencies
# -----------------------------
word_count = Counter()
for doc in Docs:
    word_count.update(tokenize(doc.content))


# -----------------------------
# Top 10 Stop Words
# -----------------------------
stop_words = [word for word, freq in word_count.most_common(10)]
print("\nTop 10 Stop Words:")
print(stop_words)


# -----------------------------
# Remove Stop Words
# -----------------------------
filtered_index = {}
for word, posting in inverted_index.items():
    if word not in stop_words:
        filtered_index[word] = posting

print("\nIndex Size Before:", len(inverted_index))
print("Index Size After :", len(filtered_index))

print("\nFILTERED INVERTED INDEX")
print(filtered_index)


# -----------------------------
# Boolean Query Processing
# -----------------------------
def process_query(query, index):
    tokens = query.lower().split()
    result = set(index.get(tokens[0], []))
    # print(result)
    i = 1
    while i < len(tokens):
        operator = tokens[i].upper()
        term = tokens[i + 1]
        docs = set(index.get(term, []))
        if operator == "AND":
            result = result.intersection(docs)
        elif operator == "OR":
            result = result.union(docs)
        i += 2
    return sorted(result)


# -----------------------------
# Sample Queries
# -----------------------------
queries = [
    # "123 AND science",
    "computer AND science",
    "python AND data",
    "computer OR python",
    "artificial AND intelligence",
    "computer AND science OR intelligence"
]

print("\nBOOLEAN QUERY RESULTS")

for q in queries:
    print(f"\nQuery : {q}")
    result = process_query(q, inverted_index)  # or filtered_index

    if result:
        print("Documents:", result)
    else:
        print("No matching documents.")

INVERTED INDEX
{'apple': ['D1', 'D2'], 'fruit': ['D1'], 'fun': ['D1'], 'science': ['D1', 'D3', 'D4', 'D5'], 'computer': ['D1', 'D2', 'D3', 'D7', 'D9'], 'a': ['D1', 'D4', 'D5'], 'is': ['D1', 'D2', 'D4', 'D5'], 'healthy': ['D1'], 'makes': ['D2'], 'engineering': ['D2', 'D3'], 'interesting': ['D2'], 'computers': ['D2'], 'are': ['D3', 'D6'], 'related': ['D3'], 'artificial': ['D3', 'D4', 'D10'], 'uses': ['D3'], 'and': ['D3', 'D5', 'D6', 'D7', 'D8', 'D9', 'D10'], 'intelligence': ['D3', 'D4', 'D10'], 'data': ['D4', 'D5', 'D10'], 'machine': ['D4', 'D5'], 'growing': ['D4'], 'learning': ['D4', 'D5'], 'of': ['D4'], 'branch': ['D4'], 'for': ['D5', 'D6', 'D8'], 'language': ['D5'], 'programming': ['D5', 'D7'], 'popular': ['D5'], 'python': ['D5', 'D6'], 'web': ['D6'], 'software': ['D6', 'D9'], 'applications': ['D6', 'D8'], 'development': ['D6'], 'java': ['D6'], 'widely': ['D6'], 'used': ['D6'], 'college': ['D7'], 'algorithms': ['D7'], 'students': ['D7'], 'learn': ['D7'], 'networks': ['D7'], 'in': ['D7